In [1]:
!pip install firebase-admin

  Using cached firebase_admin-7.1.0-py3-none-any.whl (137 kB)
  Using cached PyJWT-2.10.1-py3-none-any.whl (22 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.2/377.2 kB 6.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.4/303.4 kB 22.3 MB/s eta 0:00:00
  Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 8.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.9/152.9 kB 470.5 kB/s eta 0:00:00a 0:00:01
  Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 11.9 MB/s eta 0:00:00
  Using cached h2-4.3.0-py3-none-any.whl (61 kB)
  Using cached h11-0.16.0-py3-none-any.whl (37 kB)
  Using cached requests-2.32.5-py3-none-any.whl (64 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.8/83.8 kB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import json
import re
import firebase_admin
from firebase_admin import credentials, firestore

# 1. Inicialização
SERVICE_ACCOUNT = "/Users/leosaracino/Documents/Pessoal/Faculdade/TCC/motiva-8b82f-firebase-adminsdk-fbsvc-14d8d2b5e8.json"
try:
    firebase_admin.get_app()
except ValueError:
    cred = credentials.Certificate(SERVICE_ACCOUNT)
    firebase_admin.initialize_app(cred)

db = firestore.client()

# 2. Carrega e normaliza o JSON
with open('exercises.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

if isinstance(data, list):
    exercises = data
elif isinstance(data, dict) and 'movimentos' in data and isinstance(data['movimentos'], dict):
    exercises = list(data['movimentos'].values())
elif isinstance(data, dict):
    exercises = list(data.values())
else:
    raise ValueError("Formato JSON não suportado. Deve ser array de objetos ou dict de dicts.")

# 3. Gerador de IDs (snake_case) caso falte
def slugify(name: str) -> str:
    s = name.strip().lower()
    s = re.sub(r'[^a-z0-9]+', '_', s)
    return s.strip('_')

# 4. Importação com verificação de existência
for ex in exercises:
    # garante displayName
    disp = ex.get('displayName')
    if not disp:
        print("⚠️  Exercício sem displayName, pulando:", ex)
        continue

    # ID do doc
    doc_id = ex.get('name') or slugify(disp)
    ex['name'] = doc_id

    doc_ref = db.collection('movimentos').document(doc_id)
    if doc_ref.get().exists:
        print(f"⏭️  Já existe, pulando: {doc_id}")
        continue

    # grava
    try:
        doc_ref.set(ex)
        print(f"✔️  Importado: {doc_id}")
    except Exception as e:
        print(f"❌ Falha ao importar {doc_id}: {e}")

print("✨ Importação concluída!")


/Users/leosaracino/.pyenv/versions/3.10.13/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.13) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


FileNotFoundError: [Errno 2] No such file or directory: '/Users/leosaracino/Documents/Pessoal/Faculdade/TCC/motiva-8b82f-firebase-adminsdk-fbsvc-14d8d2b5e8.json'